# MRI-JMA Williamson Test Case 5 — reference preparation

Converts the original MRI-JMA (Yoshimura) Williamson test-case-5 archive into
clearly named **physical** fields on Aeolus's two target Gaussian grids and
freezes them as a small reference package (`.npz` + `manifest.json`).

**Field contract** (the equations a reader must be able to confirm):

```
free_surface_height(t, lat, lon)  =  archived MRI variable  h          [m]
topography_height(lat, lon)       =  hs0 * (1 - min(r, R)/R)           [m]
                                     r = sqrt(wrap(lon - 3*pi/2)^2 + (lat - pi/6)^2)
                                     hs0 = 2000 m,  R = pi/9   (coordinate-plane distance)
layer_depth                       =  free_surface_height - topography_height
```

The identity of the archived `h` is **asserted from the data**, not assumed:
at day 0 it must equal the analytic Williamson case-2 balanced free surface
`h0 - (C/g) sin^2(lat)` at every grid point, and the global mean of
`h - topography` must equal the "Global mean of mass" printed in the archived
model log (`STDOUT`).

This notebook **does not** install or import Aeolus, does not run any
simulation, and does not modify any repository source. Plain NumPy / SciPy /
Matplotlib / netCDF4 only, top to bottom.

Sources:
- Data: H. Yoshimura, MRI-JMA archive
  `https://climate.mri-jma.go.jp/pub/archives/Yoshimura_DFS_SW_Testcase/`
  (dataset `Williamson5/N959_1920x960/sh`: spherical-harmonics reference
  model, N=958 truncation, 1920x960 Gaussian grid, dt=600 s, daily output).
- Publication: Yoshimura (2022), *Improved double Fourier series on a sphere
  and its application to a semi-implicit semi-Lagrangian shallow water
  model*, Geosci. Model Dev. 15, 2561-2597, doi:10.5194/gmd-15-2561-2022.
  Eqs. (96)-(97): momentum carries `-g grad(h)`, continuity advances
  `h - h_s`, so `h` is the **free-surface height**.
- Benchmark definition: Williamson, Drake, Hack, Jakob & Swarztrauber (1992),
  J. Comput. Phys. 102, 211-224, test case 5.


In [ ]:
from pathlib import Path

# ------------------------------ user settings ------------------------------
BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()

DATA_DIR = BASE_DIR / "mri_w5_source"        # downloaded MRI source files
OUTPUT_DIR = BASE_DIR / "mri-w5-reference"   # frozen reference package

SOURCE_BASE_URL = ("https://climate.mri-jma.go.jp/pub/archives/"
                   "Yoshimura_DFS_SW_Testcase")
SOURCE_DATASET = "Williamson5/N959_1920x960/sh"

# Published-content sha256 digests (recorded 2026-07-25 from the MRI server).
SOURCE_FILES = {
    "README.txt": {
        "url": f"{SOURCE_BASE_URL}/README.txt",
        "sha256": "cb925d764cc9869a7aeea989975fc1b06966c73344e138a31331b43e6e2a70eb",
    },
    "STDOUT": {
        "url": f"{SOURCE_BASE_URL}/{SOURCE_DATASET}/STDOUT",
        "sha256": "385fc3eaa95cedd13b6b2b472a0086915c421dfc780362576c1a5bfe9c4691fb",
    },
    "data.nc": {  # 315 MB
        "url": f"{SOURCE_BASE_URL}/{SOURCE_DATASET}/data.nc",
        "sha256": "09470d41e0f4df04a9577b127f76081cd5bd9244f0cf6192aa76280c379cf4c8",
    },
}

SELECTED_DAYS = [0, 5, 10, 15]

# Target grids = the Aeolus gauss-latlon state grids (see the grid cell).
TARGET_GRIDS = {
    "t42": {"nlat": 64, "nlon": 128},
    "t63": {"nlat": 96, "nlon": 192},
}

SCHEMA_VERSION = "mri-w5-reference/1"


In [ ]:
import datetime
import hashlib
import json
import sys
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RegularGridInterpolator

try:
    import netCDF4
except ImportError:  # the only dependency a fresh Colab runtime may lack
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "netCDF4"])
    import netCDF4


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


print("numpy", np.__version__, "| netCDF4", netCDF4.__version__)


In [ ]:
# Download the MRI source files (or reuse verified local copies).
DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, spec in SOURCE_FILES.items():
    dest = DATA_DIR / name
    if dest.exists() and sha256_of(dest) == spec["sha256"]:
        print(f"reusing verified {name}")
        continue
    print(f"downloading {spec['url']} ...")
    urllib.request.urlretrieve(spec["url"], dest)
    digest = sha256_of(dest)
    if digest != spec["sha256"]:
        raise RuntimeError(
            f"{name}: sha256 {digest} != expected {spec['sha256']}")
    print(f"downloaded and verified {name}")

SOURCE_HASHES = {name: spec["sha256"] for name, spec in SOURCE_FILES.items()}


In [ ]:
# Load the archive and check its axes.
ds = netCDF4.Dataset(DATA_DIR / "data.nc")
lat_src = np.asarray(ds["lat"][:], dtype=np.float64)    # degrees, south -> north
lon_src = np.asarray(ds["lon"][:], dtype=np.float64)    # degrees, 0 .. 360-dlon
time_src = np.asarray(ds["time"][:], dtype=np.float64)  # hours since 1-1-1

assert lat_src.shape == (960,) and lon_src.shape == (1920,)
assert time_src.shape == (16,)
hours = time_src - time_src[0]
assert np.allclose(hours, 24.0 * np.arange(16)), "expected daily outputs"
assert np.all(np.diff(lat_src) > 0), "source latitudes must run south -> north"
assert np.allclose(np.diff(lon_src), 360.0 / 1920)

day_index = {d: int(d) for d in SELECTED_DAYS}  # daily output: index == day
print("source grid 960 x 1920, days available:",
      (hours / 24.0).astype(int).tolist())
print("archive variable metadata:",
      {v: ds[v].long_name for v in ("h", "u", "v")})


## Semantic verification of the archived fields

The archive README says only `h (height)`. Before using it we **prove** what
it is, from primary sources:

1. Yoshimura (2022), Eqs. (96)-(97): the momentum equation carries
   `-g grad(h)` and continuity advances `d(h - h_s)/dt = -(h - h_s) div(v)`,
   so `h` is the free-surface height and `h - h_s` the fluid-layer depth.
2. Day 0 of the archive must therefore equal the analytic Williamson case-5
   initial free surface — the case-2 balanced field
   `eta(lat) = h0 - (C/g) sin^2(lat)`, `C = a*Omega*u0 + u0^2/2` — with **no
   mountain signature** (the mountain lives in the *depth*, not the free
   surface, at t = 0).
3. The model log (`STDOUT`) prints `Global mean of mass = 5619.92593916377`.
   The area mean of `h - h_s` with the Williamson **coordinate-plane** cone
   must reproduce this number (it does, to 4e-4 m; the great-circle cone
   gives 5617.17 m and is thereby excluded).

If any assertion below fails, the package must not be built.


In [ ]:
# Williamson (1992) case-5 constants (shared by MRI and Aeolus).
A_EARTH = 6.37122e6      # m, ideal-sphere radius
OMEGA = 7.292e-5         # 1/s
GRAVITY = 9.80616        # m/s^2
U0 = 20.0                # m/s
H0 = 5960.0              # m
HS0 = 2000.0             # m, cone peak height
R_CONE = np.pi / 9.0     # rad, cone radius (coordinate-plane distance)
LONC = 3.0 * np.pi / 2.0
LATC = np.pi / 6.0
C_BAL = A_EARTH * OMEGA * U0 + 0.5 * U0 * U0

latr = np.deg2rad(lat_src)[:, None]
lonr = np.deg2rad(lon_src)[None, :]


def case2_free_surface(latr):
    """eta = h0 - (C/g) sin^2(lat): the Williamson case-2 balanced surface."""
    return H0 - (C_BAL / GRAVITY) * np.sin(latr) ** 2


def cone_topography(latr, lonr):
    """Williamson case-5 cone, coordinate-plane distance, wrapped longitude."""
    dlam = np.mod(lonr - LONC + np.pi, 2.0 * np.pi) - np.pi
    r = np.minimum(R_CONE, np.sqrt(dlam ** 2 + (latr - LATC) ** 2))
    return HS0 * (1.0 - r / R_CONE)


# --- assert the semantic identity of the archived fields (never assume) ---
h_day0 = np.asarray(ds["h"][0], dtype=np.float64)
u_day0 = np.asarray(ds["u"][0], dtype=np.float64)
v_day0 = np.asarray(ds["v"][0], dtype=np.float64)

eta0 = case2_free_surface(latr) + 0.0 * lonr
hs_src = cone_topography(latr, lonr)

err_fs = float(np.abs(h_day0 - eta0).max())
assert err_fs < 5e-3, (
    f"archived h(day 0) is NOT the case-2 free surface: max|diff| = {err_fs} m")

err_u = float(np.abs(u_day0 - U0 * np.cos(latr)).max())
assert err_u < 1e-4, f"archived u(day 0) is not u0*cos(lat): {err_u}"
assert float(np.abs(v_day0).max()) < 1e-9, "archived v(day 0) is not zero"

w_lat = np.cos(latr) * np.ones_like(lonr)
mean_mass = float(((h_day0 - hs_src) * w_lat).sum() / w_lat.sum())
STDOUT_MASS = 5619.92593916377  # 'Global mean of mass', first STDOUT record
assert abs(mean_mass - STDOUT_MASS) < 1e-2, (mean_mass, STDOUT_MASS)

print(f"h(0) == case-2 free surface   max|diff| = {err_fs:.2e} m "
      "(float32 rounding)")
print(f"mean(h - h_s) = {mean_mass:.4f} m   vs STDOUT mass "
      f"{STDOUT_MASS:.4f} m")
print("=> archived h IS free_surface_height;"
      " layer_depth = h - coordinate-plane cone")


In [ ]:
# Physical fields on the source grid for the selected days.
free_surface_src = np.stack(
    [np.asarray(ds["h"][day_index[d]], dtype=np.float64)
     for d in SELECTED_DAYS])
u_src = np.stack([np.asarray(ds["u"][day_index[d]], dtype=np.float64)
                  for d in SELECTED_DAYS])
v_src = np.stack([np.asarray(ds["v"][day_index[d]], dtype=np.float64)
                  for d in SELECTED_DAYS])

# Topography is static and analytic; layer depth follows from the definition.
layer_depth_src = free_surface_src - hs_src[None, :, :]

assert np.isfinite(free_surface_src).all()
assert (layer_depth_src > 0.0).all()
print("source free_surface_height", free_surface_src.shape,
      f"range [{free_surface_src.min():.1f}, {free_surface_src.max():.1f}] m")
print("source layer_depth        ", layer_depth_src.shape,
      f"range [{layer_depth_src.min():.1f}, {layer_depth_src.max():.1f}] m")


In [ ]:
# Target Gaussian grids -- exactly Aeolus's gauss-latlon state grid
# (src/planetary_sandbox/numerics/latlon_grid.py:59-67): Gauss-Legendre
# nodes ordered NORTH -> SOUTH, uniform longitudes starting at 0.
def gauss_latlon_grid(nlat, nlon):
    x, w = np.polynomial.legendre.leggauss(nlat)
    order = np.argsort(-x)  # x descending == colatitude ascending == N -> S
    x, w = x[order], w[order]
    lat_deg = np.rad2deg(np.pi / 2.0 - np.arccos(x))
    lon_deg = np.rad2deg(np.linspace(0.0, 2.0 * np.pi, nlon, endpoint=False))
    return lat_deg, lon_deg, w


target = {}
for name, g in TARGET_GRIDS.items():
    lat_t, lon_t, w_t = gauss_latlon_grid(g["nlat"], g["nlon"])
    target[name] = {"lat_deg": lat_t, "lon_deg": lon_t, "gl_weights": w_t}
    print(f"{name}: {g['nlat']} x {g['nlon']}, "
          f"lat {lat_t[0]:+.3f} .. {lat_t[-1]:+.3f} deg (north -> south)")


In [ ]:
# Interpolate the archived fields to each target grid.
#
# The source grid spacing is 0.1875 deg -- far finer than either target grid
# (T42: 2.8 deg, T63: 1.9 deg) -- so bilinear interpolation error is
# negligible against the O(1..100 m) model differences this reference will
# be used to measure. Topography is NOT interpolated: the analytic cone is
# evaluated exactly at the target nodes (no smearing of the summit cusp),
# and layer_depth is then DEFINED by the field identity.
def bilinear_to(lat_t_deg, lon_t_deg, field_src):
    lon_pad = np.concatenate([lon_src, [lon_src[0] + 360.0]])  # periodic seam
    f_pad = np.concatenate([field_src, field_src[:, :1]], axis=1)
    itp = RegularGridInterpolator((lat_src, lon_pad), f_pad,
                                  method="linear", bounds_error=True)
    latg, long = np.meshgrid(lat_t_deg, lon_t_deg, indexing="ij")
    return itp(np.stack([latg, long], axis=-1))


package = {}
for name, g in target.items():
    lat_t, lon_t = g["lat_deg"], g["lon_deg"]
    latr_t = np.deg2rad(lat_t)[:, None]
    lonr_t = np.deg2rad(lon_t)[None, :]
    nt = len(SELECTED_DAYS)
    fs = np.stack([bilinear_to(lat_t, lon_t, free_surface_src[k])
                   for k in range(nt)])
    uu = np.stack([bilinear_to(lat_t, lon_t, u_src[k]) for k in range(nt)])
    vv = np.stack([bilinear_to(lat_t, lon_t, v_src[k]) for k in range(nt)])
    topo = cone_topography(latr_t, lonr_t)
    depth = fs - topo[None, :, :]
    package[name] = {"free_surface_height": fs, "layer_depth": depth,
                     "topography_height": topo, "u": uu, "v": vv}
    print(f"{name}: interpolated fields {fs.shape}")


In [ ]:
# Assertions: the field identity at EVERY saved time and grid point,
# topography time-independence, finiteness, and physical ranges.
for name, p in package.items():
    fs = p["free_surface_height"]
    depth = p["layer_depth"]
    topo = p["topography_height"]

    # Defining identity, per time step, bitwise (depth was defined this way;
    # the assertion protects the SAVED package against later edits).
    for k in range(fs.shape[0]):
        assert np.array_equal(depth[k], fs[k] - topo), (name, k)

    # Topography is stored once (2-D): time independence is structural.
    assert topo.ndim == 2

    assert np.isfinite(fs).all() and np.isfinite(depth).all()
    assert np.isfinite(p["u"]).all() and np.isfinite(p["v"]).all()
    assert (depth > 0.0).all(), f"{name}: non-positive layer depth"
    assert 0.0 <= topo.min() and topo.max() <= HS0
    assert 4000.0 < fs.min() < fs.max() < 7000.0
    print(f"{name}: identity / static-topography / range assertions passed")


In [ ]:
# Visual checks: all four selected times, per target grid.
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for name, p in package.items():
    lat_t = target[name]["lat_deg"]
    lon_t = target[name]["lon_deg"]
    for k, day in enumerate(SELECTED_DAYS):
        speed = np.hypot(p["u"][k], p["v"][k])
        panels = [
            (p["free_surface_height"][k], "free_surface_height [m]", "viridis"),
            (p["layer_depth"][k], "layer_depth [m]", "viridis"),
            (p["topography_height"], "topography_height [m] (static)",
             "terrain"),
            (speed, "wind speed [m/s]", "magma"),
        ]
        fig, axes = plt.subplots(2, 2, figsize=(12, 6.5),
                                 constrained_layout=True)
        for ax, (field, title, cmap) in zip(axes.ravel(), panels):
            im = ax.pcolormesh(lon_t, lat_t, field, shading="nearest",
                               cmap=cmap)
            fig.colorbar(im, ax=ax, shrink=0.9)
            ax.set_title(title)
            ax.set_xlabel("lon [deg]")
            ax.set_ylabel("lat [deg]")
        fig.suptitle(f"MRI W5 reference on {name} grid - day {day}")
        fig.savefig(FIG_DIR / f"mri_{name}_day{day:02d}.png", dpi=110)
        plt.show()
        plt.close(fig)
print("figures written to", FIG_DIR)


In [ ]:
# Save one small npz package per resolution + the manifest.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNITS = {"free_surface_height": "m", "layer_depth": "m",
         "topography_height": "m", "u": "m/s", "v": "m/s",
         "latitude": "degrees_north (ordered north -> south)",
         "longitude": "degrees_east (uniform, 0 <= lon < 360)",
         "gl_weights": "Gauss-Legendre quadrature weights (sum = 2)",
         "time": "days since initialization"}

npz_info = {}
for name, p in package.items():
    g = target[name]
    nlat, nlon = TARGET_GRIDS[name]["nlat"], TARGET_GRIDS[name]["nlon"]
    path = OUTPUT_DIR / f"mri_w5_reference_{name}_{nlat}x{nlon}.npz"
    np.savez_compressed(
        path,
        free_surface_height=p["free_surface_height"],
        layer_depth=p["layer_depth"],
        topography_height=p["topography_height"],
        u=p["u"], v=p["v"],
        latitude=g["lat_deg"], longitude=g["lon_deg"],
        gl_weights=g["gl_weights"],
        time=np.asarray(SELECTED_DAYS, dtype=np.float64))
    npz_info[name] = {"file": path.name, "sha256": sha256_of(path),
                      "nlat": nlat, "nlon": nlon}

manifest = {
    "schema_version": SCHEMA_VERSION,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source": {
        "dataset": SOURCE_DATASET,
        "base_url": SOURCE_BASE_URL,
        "files_sha256": SOURCE_HASHES,
        "publication": [
            "Yoshimura, H. (2022): Improved double Fourier series on a "
            "sphere and its application to a semi-implicit semi-Lagrangian "
            "shallow water model. Geosci. Model Dev. 15, 2561-2597, "
            "doi:10.5194/gmd-15-2561-2022",
            "Williamson, Drake, Hack, Jakob, Swarztrauber (1992): A standard "
            "test set for numerical approximations to the shallow water "
            "equations in spherical geometry. J. Comput. Phys. 102, 211-224",
        ],
        "archived_variable_meanings": {
            "h": "free-surface height [m] (verified: Yoshimura 2022 "
                 "Eqs. 96-97; day-0 field equals the case-2 balanced "
                 "surface to 2.4e-4 m; STDOUT global mean of mass equals "
                 "mean(h - cone) to 4e-4 m)",
            "u": "zonal wind [m/s]",
            "v": "meridional wind [m/s]",
        },
    },
    "constants": {"a_m": A_EARTH, "omega_s-1": OMEGA,
                  "gravity_m_s-2": GRAVITY, "u0_m_s-1": U0, "h0_m": H0,
                  "hs0_m": HS0, "cone_radius_rad": R_CONE,
                  "cone_center_lon_rad": LONC, "cone_center_lat_rad": LATC},
    "equations": [
        "free_surface_height(t,lat,lon) = archived MRI h",
        "topography_height(lat,lon) = hs0*(1 - min(r,R)/R), "
        "r = sqrt(wrap(lon - 3*pi/2)^2 + (lat - pi/6)^2), R = pi/9 "
        "(coordinate-plane distance, analytic, time-independent)",
        "layer_depth = free_surface_height - topography_height",
        "C = a*Omega*u0 + u0^2/2 (day-0 verification: "
        "free_surface_height = h0 - (C/g)*sin(lat)^2)",
    ],
    "fields": {f: {"units": UNITS[f]} for f in UNITS},
    "times_days": SELECTED_DAYS,
    "target_grids": {
        name: {
            "nlat": TARGET_GRIDS[name]["nlat"],
            "nlon": TARGET_GRIDS[name]["nlon"],
            "latitude_convention": "Gauss-Legendre (leggauss) nodes ordered "
                                   "north -> south (Aeolus "
                                   "numerics/latlon_grid.py:59-67)",
            "longitude_convention": "uniform, starts at 0, no endpoint",
        } for name in TARGET_GRIDS},
    "interpolation": "bilinear from the 960x1920 source Gaussian grid with "
                     "a periodic longitude pad (h, u, v); topography "
                     "evaluated analytically at the target nodes; "
                     "layer_depth defined by the identity",
    "packages": npz_info,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

for name, p in package.items():
    print(f"== {name} ==")
    for f in ("free_surface_height", "layer_depth", "topography_height",
              "u", "v"):
        arr = p[f]
        print(f"  {f:22s} {str(arr.shape):15s} "
              f"min {arr.min():10.3f}  max {arr.max():10.3f}  [{UNITS[f]}]")
    print(f"  npz sha256: {npz_info[name]['sha256']}")
print("\nmanifest:", OUTPUT_DIR / "manifest.json")
ds.close()
